# Dónde construir: estaciones para una alcaldía nueva

## Contexto

Una nueva administración acaba de asumir la Alcaldía de Medellín. Dentro del plan de desarrollo se aprobó una partida para construir estaciones que atiendan tres tipos de incidentes:

| Tipo de incidente | Estación que lo atiende |
|---|---|
| Hurto | CAI de policía |
| Accidente de tránsito | Estación de tránsito y grúas |
| Incendio o emergencia | Estación de bomberos |

La Secretaría de Planeación entrega un histórico de incidentes de los últimos años, la red vial de la ciudad y una lista de lotes disponibles donde se podría construir. Cada estación tiene un costo que depende de su tipo y de la zona del lote, y solo atiende con oportunidad los incidentes de su tipo que estén a cierta distancia por las calles.

El presupuesto no alcanza para cubrir toda la ciudad. Tu equipo debe decidir **qué estaciones construir y en qué lotes**, de manera que la ciudad quede lo mejor atendida posible sin exceder el presupuesto, y presentar la propuesta al concejo con un mapa, el costo y los casos que se cubrirían.

La red vial corresponde a las calles reales de Medellín. El histórico de incidentes, los lotes y los costos son simulados para el curso.

## Los datos

Cada instancia está en una carpeta con cinco archivos:

- **`nodos.csv`**: intersecciones y puntos de la red. Columnas `id`, `lat`, `lon`.
- **`aristas.csv`**: tramos de calle. Columnas `origen`, `destino`, `longitud_m`, `nombre`. Cada fila es un tramo que se puede recorrer **solo de `origen` a `destino`**. Una calle de doble sentido aparece dos veces, una por cada dirección. Algunos tramos no tienen nombre.
- **`lotes.csv`**: lotes disponibles. Columnas `lote` (código), `nodo` (intersección donde está el lote) y `factor_zona`.
- **`incidentes.csv`**: histórico. Columnas `id`, `nodo` (intersección donde ocurrió), `tipo` (`hurto`, `accidente` o `incendio`) y `fecha_hora`.
- **`parametros.json`**: presupuesto total y, para cada tipo de incidente, el nombre de la estación, su costo base, su radio de cobertura y su peso de prioridad.

Los parámetros de las tres instancias son los mismos salvo el presupuesto:

| Tipo | Estación | Costo base (millones) | Radio de cobertura | Peso |
|---|---|---|---|---|
| hurto | CAI | 1.000 | 1.000 m | 1 |
| accidente | Tránsito y grúas | 2.000 | 2.000 m | 3 |
| incendio | Bomberos | 5.000 | 3.000 m | 5 |

### Las tres instancias

| Instancia | Descripción | Nodos | Tramos | Incidentes | Lotes | Presupuesto (millones) |
|---|---|---|---|---|---|---|
| `pequena` | Barrio ficticio | 9 | 22 | 18 | 5 | 6.000 |
| `comuna` | Recorte de la red real alrededor de Laureles | 2.893 | 6.782 | 913 | 15 | 12.000 |
| `ciudad` | Red vial completa de Medellín | 27.568 | 67.330 | 5.300 | 80 | 40.000 |

### Descripción del barrio de la instancia `pequena`

El barrio es una cuadrícula de 3 × 3 intersecciones numeradas de 0 a 8. Vista como un mapa con el norte hacia arriba:

```
 6 ── 7 ── 8        fila de arriba:  Calle 3
 │    │    ↑
 3 ── 4 →─ 5        fila del medio:  Calle 2
 │    │    │
 0 ── 1 ── 2        fila de abajo:   Calle 1

Carrera A (0-3-6)   Carrera B (1-4-7)   Carrera C (2-5-8)
```

Todas las calles son de doble sentido excepto dos tramos: en la Calle 2 solo se puede ir **de 4 a 5**, y en la Carrera C solo se puede ir **de 5 a 8**. Cada tramo mide entre 900 m y 990 m; las longitudes exactas están en `aristas.csv`.

Los lotes disponibles están en las intersecciones 1, 3, 5, 6 y 8:

| lote | nodo | factor_zona |
|---|---|---|
| L1 | 1 | 1.1 |
| L2 | 3 | 1.5 |
| L3 | 5 | 1.4 |
| L4 | 6 | 1.5 |
| L5 | 8 | 1.5 |

Los 18 incidentes del histórico se distribuyen así (H = hurtos, A = accidentes, I = incendios):

| Nodo | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| Incidentes | — | H1 | H1 A2 I1 | H3 | H2 A1 | A1 | A1 I1 | H1 | H1 A1 I1 |

## Reglas del modelo

1. **Costo.** Construir una estación de un tipo en un lote cuesta `costo_base del tipo × factor_zona del lote`, redondeado al millón más cercano. Por ejemplo, un CAI en el lote L1 de la instancia `pequena` cuesta 1.000 × 1.1 = 1.100 millones.
2. **Un lote, una estación.** En cada lote se construye a lo sumo una estación. Sí se pueden construir varias estaciones del mismo tipo en lotes distintos.
3. **Presupuesto.** La suma de los costos de las estaciones construidas no puede superar el presupuesto de la instancia.
4. **Cobertura.** Un incidente queda cubierto si existe una estación **de su mismo tipo** tal que la distancia más corta por la red vial **desde el nodo de la estación hasta el nodo del incidente** es menor o igual al radio de cobertura de ese tipo. La distancia se mide respetando el sentido de los tramos, porque es el vehículo de la estación el que se desplaza hasta el incidente.
5. **Valor de una propuesta.** Es la suma de los pesos de los incidentes cubiertos. Un incidente cubierto por varias estaciones cuenta una sola vez. Por ejemplo, una propuesta que cubre 10 hurtos, 4 accidentes y 1 incendio vale 10 × 1 + 4 × 3 + 1 × 5 = 27.

### Supuestos

- La red vial es una aproximación: se retiraron ciclorrutas, senderos y pasos peatonales identificables por su nombre.
- Los vehículos de emergencia pueden circular por los carriles de Metroplús, que hacen parte de la red.
- Todos los incidentes de un mismo tipo tienen la misma prioridad, sin importar su fecha.

## Restricciones de implementación

- La solución se escribe en Python.
- Se pueden usar `csv`, `json` o `pandas` **solo para leer los archivos**, y `heapq`.
- Para el mapa y la interfaz gráfica se puede usar cualquier librería de visualización o de interfaces (por ejemplo `folium`, `matplotlib`, `streamlit` o `tkinter`), siempre que no se use para calcular rutas, distancias ni combinaciones.
- No se permite usar `networkx`, `osmnx`, `scipy`, `itertools` ni ninguna librería que calcule rutas, distancias en grafos o combinaciones.

## Cronograma y entregables

| Semana | Entrega |
|---|---|
| 1 | Parte 1 |
| 2 | Parte 2 |
| 3 | Parte 3 |
| 4 | Parte 4 y sustentación |

Cada entrega se hace sobre este mismo notebook. Las partes posteriores pueden reutilizar el código de las anteriores.

### Sustentación

En la sustentación se entregará a cada equipo una nueva distribución de lotes y un nuevo histórico de incidentes, con el mismo formato de archivos descrito en la sección "Los datos". El equipo deberá ejecutar su solución sobre esos datos durante la sustentación y ajustarla si es necesario, así que conviene que el código lea cualquier instancia desde sus archivos y no dependa de valores particulares de las instancias entregadas.

### Nota: feria de proyectos de ingeniería

Los equipos que quieran presentar esta práctica en la feria de proyectos de ingeniería deben, además de todo lo anterior, desarrollar una **aplicación interactiva con interfaz gráfica de usuario**, por ejemplo una que permita elegir la instancia, cambiar el presupuesto o los pesos de prioridad y ver la propuesta resultante sobre el mapa. Sin esta aplicación el proyecto no podrá competir.

El equipo ganador de la feria quedará **eximido del examen final**.

## Preparación del entorno

El código de la solución vive en el paquete `estaciones/` (solo `csv`, `json`, `heapq`, `math`, `time`, `pathlib`, `typing`; **sin** `networkx`, `scipy` ni `itertools`) y las visualizaciones en `visual/` (`folium`, `matplotlib`, `plotly`). El notebook importa, ejecuta y valida contra los casos oficiales con `assert`.

```
estaciones/  datos.py · grafo.py · alcance.py · evaluacion.py · heuristicas.py · exacto.py · reportes.py
visual/      mapas.py · graficas.py · informe.py
contraejemplos/  generar_contraejemplos.py + 4 instancias
app_feria_streamlit.py   ← aplicación interactiva para la feria
```

In [1]:
import sys, time, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from IPython.display import IFrame, Image, display, Markdown

from estaciones import (cargar_instancia, calcular_alcance, HEURISTICAS, heuristica_base, eficiencia_marginal,
                        lookahead_1_paso, hibrida_adaptativa, busqueda_local_swap, mejor_propuesta_exacta)
from estaciones.reportes import texto_tabla, tabla_comparativa, demostracion_ciudad, nodos_candidatos
os.makedirs("salidas", exist_ok=True)

## Parte 1. Alcance de cada estación posible

Para una instancia cualquiera, calcula para **cada lote y cada tipo de estación**:

- el costo de construir esa estación en ese lote, y
- la lista de `id` de los incidentes que esa estación cubriría por sí sola, según la regla 4.

### Entrada

El nombre de una instancia (`pequena`, `comuna` o `ciudad`). El programa lee los cinco archivos de su carpeta.

### Salida

Para cada par (lote, tipo): su costo en millones y la lista ordenada de `id` de los incidentes que cubre.

### Restricción adicional

El cálculo para la instancia `ciudad`, con sus 80 lotes y 3 tipos, debe terminar en menos de un minuto.

In [2]:
casos_ej_001_parte1 = [
    {
        "entrada": {"instancia": "pequena", "lote": "L3", "tipo": "accidente"},
        "salida": {"costo": 2800, "incidentes_cubiertos": [1, 2, 4, 6]},
    },
    {
        "entrada": {"instancia": "pequena", "lote": "L2", "tipo": "hurto"},
        "salida": {"costo": 1500, "incidentes_cubiertos": [9, 10, 11, 12, 13]},
    },
]

### Solución Parte 1

**Diseño.** Se carga la red con `csv` como lista de adyacencia **dirigida** (si hay aristas paralelas se conserva la de menor longitud). Para cada lote se ejecuta **un solo Dijkstra acotado** (`heapq`) hasta el mayor radio (3.000 m): cualquier radio menor se resuelve filtrando las mismas distancias. Los incidentes se agrupan por `(tipo, nodo)` y se representan como **máscaras de bits** por tipo, lo que hace que las partes 2 y 3 operen con `|`, `&` y `.bit_count()`. El costo es `round(costo_base × factor_zona)` con redondeo half-up.

In [3]:
INSTANCIAS = ["pequena", "comuna", "ciudad"]
INST, ALC = {}, {}
for nombre in INSTANCIAS:
    t0 = time.perf_counter(); INST[nombre] = cargar_instancia(nombre); t_carga = time.perf_counter() - t0
    t0 = time.perf_counter(); ALC[nombre] = calcular_alcance(INST[nombre]); t_alc = time.perf_counter() - t0
    i = INST[nombre]
    print(f"{nombre:8s} nodos={len(i.nodos):6d} tramos={i.n_aristas:6d} lotes={len(i.lotes):3d} "
          f"incidentes={len(i.incidentes):5d} | carga {t_carga:.2f}s · alcance (Dijkstra por lote) {t_alc:.2f}s")
assert t_alc < 60, "ciudad excede el límite de 1 minuto"
print("\nCiudad cumple el límite de 60 s con amplio margen.")

pequena  nodos=     9 tramos=    22 lotes=  5 incidentes=   18 | carga 0.00s · alcance (Dijkstra por lote) 0.00s
comuna   nodos=  2893 tramos=  6782 lotes= 15 incidentes=  913 | carga 0.06s · alcance (Dijkstra por lote) 0.14s


ciudad   nodos= 27568 tramos= 67330 lotes= 80 incidentes= 5300 | carga 0.40s · alcance (Dijkstra por lote) 0.70s

Ciudad cumple el límite de 60 s con amplio margen.


In [4]:
# Salida de la Parte 1: costo y lista ordenada de incidentes cubiertos por cada (lote, tipo) — ejemplo en `pequena`
al = ALC["pequena"]
filas = [{"lote": l, "tipo": t, "costo": al.costo[(l, t)], "incidentes_cubiertos": al.cubiertos[(l, t)]}
         for l, _, _ in INST["pequena"].lotes for t in INST["pequena"].tipos]
pd.DataFrame(filas)

,lote,tipo,costo,incidentes_cubiertos
0,L1,hurto,1100,"[7, 8, 12, 13]"
1,L1,accidente,2200,"[1, 2, 3, 4]"
2,L1,incendio,5500,"[16, 17, 18]"
3,L2,hurto,1500,"[9, 10, 11, 12, 13]"
4,L2,accidente,3000,"[3, 4, 5]"
5,L2,incendio,7500,"[16, 17, 18]"
6,L3,hurto,1400,"[8, 15]"
7,L3,accidente,2800,"[1, 2, 4, 6]"
8,L3,incendio,7000,"[16, 17, 18]"
9,L4,hurto,1500,"[9, 10, 11, 14]"


In [5]:
# Validación contra casos_ej_001_parte1 (los `assert` fallan si algo no coincide)
for caso in casos_ej_001_parte1:
    e, s = caso["entrada"], caso["salida"]
    a = ALC[e["instancia"]]
    assert a.costo[(e["lote"], e["tipo"])] == s["costo"], e
    assert a.cubiertos[(e["lote"], e["tipo"])] == s["incidentes_cubiertos"], e
print("✔ Parte 1: casos oficiales OK")

✔ Parte 1: casos oficiales OK


## Parte 2. Propuestas construidas paso a paso

### 2.1 Heurística base

La Secretaría de Planeación usa hoy el siguiente procedimiento para armar una propuesta:

1. Empezar sin estaciones y con todo el presupuesto disponible.
2. Considerar todas las estaciones posibles (lote, tipo) que cumplen dos condiciones: el lote todavía está libre y su costo no supera el presupuesto restante.
3. De ellas, elegir la que **agrega más valor**, es decir, la que tiene mayor suma de pesos de los incidentes que cubre y que **todavía no están cubiertos** por las estaciones ya elegidas. Si hay empate, se elige la de menor código de lote; si el empate es dentro del mismo lote, se prefiere en el orden hurto, accidente, incendio.
4. Si ninguna estación posible agrega valor, terminar. Si no, construirla, descontar su costo, marcar el lote como ocupado y volver al paso 2.

Implementa la heurística base y aplícala a las tres instancias.

### 2.2 Heurísticas propias

Diseña al menos una heurística propia que también construya la propuesta eligiendo una estación a la vez, y que obtenga un valor mayor que la base en al menos una de las tres instancias. Aplícala a las tres instancias.

Para cada heurística, la base incluida, construye a mano una instancia pequeña (en el mismo formato de archivos, con no más de 6 nodos) en la que la heurística **no** obtenga la mejor propuesta posible. Explica qué decisión de la heurística la lleva por mal camino en ese caso.

### Salida

Para cada heurística e instancia: las estaciones elegidas (lote y tipo, ordenadas por código de lote), el costo total, la cantidad de casos cubiertos de cada tipo y el valor de la propuesta.

In [6]:
casos_ej_001_parte2 = [
    {
        "entrada": {"instancia": "pequena", "heuristica": "base"},
        "salida": {
            "estaciones": [("L1", "incendio")],
            "costo_total": 5500,
            "casos_cubiertos": {"hurto": 0, "accidente": 0, "incendio": 3},
            "valor": 15,
        },
    },
    {
        "entrada": {"instancia": "comuna", "heuristica": "base"},
        "salida": {
            "estaciones": [
                ("L05", "accidente"), ("L06", "hurto"), ("L08", "accidente"),
                ("L09", "incendio"), ("L13", "hurto"), ("L14", "hurto"),
                ("L15", "accidente"),
            ],
            "costo_total": 11950,
            "casos_cubiertos": {"hurto": 486, "accidente": 296, "incendio": 31},
            "valor": 1529,
        },
    },
]

### Solución Parte 2

**Heurística base.** Esqueleto constructivo común: en cada paso se evalúa cada `(lote, tipo)` libre y asequible, se calcula su **valor marginal** (incidentes aún no cubiertos × peso) con máscaras y se elige el mejor; el desempate oficial se obtiene recorriendo los candidatos en el orden (código de lote, hurto→accidente→incendio) y usando comparación estricta.

**Heurísticas propias** (cada una elige una estación a la vez):
1. `eficiencia_marginal`: maximiza `valor_marginal / costo`.
2. `lookahead_1_paso`: elige `c1` que maximiza `vm(c1) + max_{c2 factible tras c1} vm(c2 | c1)`.
3. `hibrida_adaptativa`: maximiza `vm / costo^0.75` mientras quede más del 25 % del presupuesto y luego **llena el remanente** con valor marginal puro.

Además se incluye como extra `busqueda_local_swap`, que mejora cualquier propuesta con movimientos 1-swap (cambiar de lote o de tipo, o quitar una y agregar otra).

In [7]:
def mostrar(nombre_inst, metodo, p):
    print(f"[{nombre_inst} · {metodo}]  valor={p.valor}  costo={p.costo_total}  casos={p.casos_cubiertos}")
    print("   estaciones:", p.estaciones)

RES = {n: {} for n in INSTANCIAS}
for n in INSTANCIAS:
    for k, f in HEURISTICAS.items():
        RES[n][k] = f(ALC[n]); mostrar(n, k, RES[n][k])
    print()

[pequena · base]  valor=15  costo=5500  casos={'hurto': 0, 'accidente': 0, 'incendio': 3}
   estaciones: [('L1', 'incendio')]
[pequena · eficiencia_marginal]  valor=19  costo=5100  casos={'hurto': 7, 'accidente': 4, 'incendio': 0}
   estaciones: [('L1', 'accidente'), ('L2', 'hurto'), ('L3', 'hurto')]
[pequena · lookahead_1_paso]  valor=19  costo=5100  casos={'hurto': 7, 'accidente': 4, 'incendio': 0}
   estaciones: [('L1', 'accidente'), ('L2', 'hurto'), ('L3', 'hurto')]
[pequena · hibrida_adaptativa]  valor=19  costo=5100  casos={'hurto': 7, 'accidente': 4, 'incendio': 0}
   estaciones: [('L1', 'accidente'), ('L2', 'hurto'), ('L3', 'hurto')]

[comuna · base]  valor=1529  costo=11950  casos={'hurto': 486, 'accidente': 296, 'incendio': 31}
   estaciones: [('L05', 'accidente'), ('L06', 'hurto'), ('L08', 'accidente'), ('L09', 'incendio'), ('L13', 'hurto'), ('L14', 'hurto'), ('L15', 'accidente')]
[comuna · eficiencia_marginal]  valor=1492  costo=11600  casos={'hurto': 532, 'accidente': 320,

[ciudad · lookahead_1_paso]  valor=7566  costo=39750  casos={'hurto': 1241, 'accidente': 1785, 'incendio': 194}
   estaciones: [('L01', 'accidente'), ('L04', 'hurto'), ('L05', 'accidente'), ('L08', 'incendio'), ('L11', 'hurto'), ('L12', 'accidente'), ('L14', 'accidente'), ('L16', 'accidente'), ('L19', 'accidente'), ('L28', 'incendio'), ('L29', 'incendio'), ('L34', 'accidente'), ('L37', 'accidente'), ('L39', 'accidente'), ('L46', 'hurto'), ('L57', 'accidente'), ('L62', 'hurto'), ('L67', 'accidente'), ('L70', 'hurto')]
[ciudad · hibrida_adaptativa]  valor=7684  costo=39800  casos={'hurto': 1468, 'accidente': 1752, 'incendio': 192}
   estaciones: [('L01', 'accidente'), ('L04', 'hurto'), ('L05', 'accidente'), ('L08', 'hurto'), ('L11', 'hurto'), ('L14', 'accidente'), ('L19', 'accidente'), ('L25', 'accidente'), ('L28', 'incendio'), ('L29', 'incendio'), ('L32', 'accidente'), ('L34', 'accidente'), ('L35', 'hurto'), ('L37', 'accidente'), ('L45', 'accidente'), ('L46', 'hurto'), ('L57', 'accident

In [8]:
# Validación contra casos_ej_001_parte2
for caso in casos_ej_001_parte2:
    e, s = caso["entrada"], caso["salida"]
    p = RES[e["instancia"]][e["heuristica"]]
    assert p.estaciones == s["estaciones"], (e, p.estaciones)
    assert p.costo_total == s["costo_total"] and p.casos_cubiertos == s["casos_cubiertos"] and p.valor == s["valor"], e
# Requisito 2.2: al menos una heurística propia supera a la base en alguna instancia
mejoras = [(n, k, RES[n][k].valor - RES[n]["base"].valor) for n in INSTANCIAS for k in HEURISTICAS if k != "base"
           if RES[n][k].valor > RES[n]["base"].valor]
assert mejoras, "ninguna heurística propia supera a la base"
print("✔ Parte 2: casos oficiales OK. Mejoras sobre la base (instancia, heurística, +valor):")
for m in mejoras: print("  ", m)

✔ Parte 2: casos oficiales OK. Mejoras sobre la base (instancia, heurística, +valor):
   ('pequena', 'eficiencia_marginal', 4)
   ('pequena', 'lookahead_1_paso', 4)
   ('pequena', 'hibrida_adaptativa', 4)
   ('comuna', 'lookahead_1_paso', 15)
   ('ciudad', 'eficiencia_marginal', 115)
   ('ciudad', 'hibrida_adaptativa', 93)


In [9]:
# Extra: búsqueda local 1-swap sobre cada heurística
filas = []
for n in INSTANCIAS:
    for k in HEURISTICAS:
        s = busqueda_local_swap(ALC[n], RES[n][k])
        assert s.valor >= RES[n][k].valor and s.costo_total <= INST[n].presupuesto
        filas.append({"instancia": n, "heurística": k, "valor": RES[n][k].valor, "valor + swap": s.valor,
                      "ganancia": s.valor - RES[n][k].valor})
pd.DataFrame(filas)

,instancia,heurística,valor,valor + swap,ganancia
0,pequena,base,15,15,0
1,pequena,eficiencia_marginal,19,19,0
2,pequena,lookahead_1_paso,19,19,0
3,pequena,hibrida_adaptativa,19,19,0
4,comuna,base,1529,1529,0
5,comuna,eficiencia_marginal,1492,1512,20
6,comuna,lookahead_1_paso,1544,1544,0
7,comuna,hibrida_adaptativa,1529,1529,0
8,ciudad,base,7591,7630,39
9,ciudad,eficiencia_marginal,7706,7784,78


### Contraejemplos (una instancia de ≤ 6 nodos por heurística)

`contraejemplos/generar_contraejemplos.py` escribe cuatro carpetas con los cinco archivos reglamentarios, ejecuta cada heurística y el solver exacto y **comprueba con `assert`** que la heurística objetivo no alcanza el óptimo. Cada carpeta tiene un `README.md` con el paso a paso. Resumen de la decisión que desvía a cada una:

| Heurística | Instancia | Decisión que la desvía |
|---|---|---|
| `base` | `contraejemplos/base` | Toma bomberos (valor 10, costo 5.000) por ser lo de mayor valor marginal y no queda presupuesto para tres tránsitos que suman 18. |
| `eficiencia_marginal` | `contraejemplos/eficiencia_marginal` | Dos CAI baratos (0,0040 valor/M) superan en cociente a bomberos (0,0030) y luego ya no cabe la estación de valor 15. |
| `lookahead_1_paso` | `contraejemplos/lookahead_1_paso` | Ve solo dos estaciones: bomberos + CAI vale 16 y tres tránsitos (18) quedan fuera del horizonte. |
| `hibrida_adaptativa` | `contraejemplos/hibrida_adaptativa` | Con exponente 0,75 el CAI (0,0281) sigue superando a bomberos (0,0252); se gastan 2.000 y sobra un remanente que no alcanza. |

In [10]:
import subprocess
out = subprocess.run([sys.executable, "contraejemplos/generar_contraejemplos.py"], capture_output=True, text=True)
print(out.stdout, out.stderr)
assert out.returncode == 0
for k in HEURISTICAS:
    inst = cargar_instancia(Path("contraejemplos") / k); al = calcular_alcance(inst)
    opt, _ = mejor_propuesta_exacta(al)
    h = HEURISTICAS[k](al)
    assert len(inst.nodos) <= 6 and h.valor < opt.valor
    print(f"{k:22s} nodos={len(inst.nodos)}  heurística={h.valor:3d}  óptimo={opt.valor:3d}")

base                   heurística= 10  óptimo= 18  OK
eficiencia_marginal    heurística=  8  óptimo= 15  OK
lookahead_1_paso       heurística= 16  óptimo= 18  OK
hibrida_adaptativa     heurística= 10  óptimo= 15  OK
 
base                   nodos=4  heurística= 10  óptimo= 18
eficiencia_marginal    nodos=3  heurística=  8  óptimo= 15
lookahead_1_paso       nodos=5  heurística= 16  óptimo= 18
hibrida_adaptativa     nodos=3  heurística= 10  óptimo= 15


## Parte 3. La mejor propuesta posible

Obtén la propuesta de **mayor valor posible** que respete todas las reglas, para las instancias `pequena` y `comuna`. Tu programa debe garantizar que no existe una propuesta mejor, no solo encontrar una buena.

Luego compara en una tabla, para las tres instancias, el valor y el tiempo de ejecución de la heurística base, de tus heurísticas propias y de la mejor propuesta posible. En la instancia `ciudad`, explica con un cálculo por qué no reportas la mejor propuesta posible.

### Salida

Para `pequena` y `comuna`: las estaciones de la mejor propuesta (ordenadas por código de lote), su costo total, los casos cubiertos de cada tipo, su valor y el tiempo que tardó en obtenerse. Además, la tabla comparativa.

In [11]:
casos_ej_001_parte3 = [
    {
        "entrada": {"instancia": "pequena"},
        "salida": {
            "estaciones": [("L1", "hurto"), ("L3", "accidente"), ("L4", "hurto")],
            "costo_total": 5400,
            "casos_cubiertos": {"hurto": 8, "accidente": 4, "incendio": 0},
            "valor": 20,
        },
    },
]

### Solución Parte 3

**Branch & Bound** (`estaciones/exacto.py`): DFS recursivo por lote; en cada lote se ramifica en *no construir* o construir cada tipo (solo los que aportan cobertura). El estado es **una sola máscara `int`** (un rango de bits por tipo) y el valor se calcula con `.bit_count()`.

- **Cota inferior inicial:** la mejor propuesta de las heurísticas de la Parte 2 (y su refinamiento con 1-swap).
- **Poda 1 (unión de sufijo):** valor de `cubierto | OR(máscaras de los lotes restantes)`, ignorando el presupuesto.
- **Poda 2 (mochila fraccionaria):** con el presupuesto restante, se ordenan las estaciones restantes por valor marginal/costo y se llena fraccionariamente; como la suma de marginales individuales ≥ marginal conjunto (submodularidad), es una cota superior válida.
- **Desempate canónico** entre óptimos: menor costo y luego orden lexicográfico. Como solo se poda cuando `cota < mejor`, se garantiza que **no existe una propuesta mejor**.

In [12]:
EXACTO = {}
for n in ("pequena", "comuna"):
    t0 = time.perf_counter(); p, st = mejor_propuesta_exacta(ALC[n]); dt = time.perf_counter() - t0
    EXACTO[n] = (p, dt)
    print(f"[{n}] ÓPTIMO valor={p.valor} costo={p.costo_total} casos={p.casos_cubiertos} tiempo={dt:.2f}s")
    print("   estaciones:", p.estaciones)
    print("   nodos explorados:", st["nodos"], "| podas unión:", st["podas_union"], "| podas mochila:", st["podas_mochila"],
          "| cota inicial (heurísticas):", st["cota_inicial"])

[pequena] ÓPTIMO valor=20 costo=5400 casos={'hurto': 8, 'accidente': 4, 'incendio': 0} tiempo=0.00s
   estaciones: [('L1', 'hurto'), ('L3', 'accidente'), ('L4', 'hurto')]
   nodos explorados: 39 | podas unión: 0 | podas mochila: 22 | cota inicial (heurísticas): 19


[comuna] ÓPTIMO valor=1574 costo=11700 casos={'hurto': 519, 'accidente': 300, 'incendio': 31} tiempo=1.33s
   estaciones: [('L03', 'accidente'), ('L06', 'hurto'), ('L09', 'hurto'), ('L11', 'hurto'), ('L13', 'accidente'), ('L14', 'hurto'), ('L15', 'incendio')]
   nodos explorados: 48079 | podas unión: 10578 | podas mochila: 23751 | cota inicial (heurísticas): 1544


In [13]:
# Validación contra casos_ej_001_parte3 + el óptimo nunca es peor que ninguna heurística
for caso in casos_ej_001_parte3:
    s = caso["salida"]; p = EXACTO[caso["entrada"]["instancia"]][0]
    assert p.estaciones == s["estaciones"] and p.costo_total == s["costo_total"]
    assert p.casos_cubiertos == s["casos_cubiertos"] and p.valor == s["valor"]
for n in EXACTO:
    assert all(EXACTO[n][0].valor >= RES[n][k].valor for k in HEURISTICAS)
print("✔ Parte 3: caso oficial OK y óptimo ≥ todas las heurísticas")

✔ Parte 3: caso oficial OK y óptimo ≥ todas las heurísticas


In [14]:
# Tabla comparativa: valor y tiempo de cada método en las 3 instancias
filas = tabla_comparativa(ALC, HEURISTICAS)
df = pd.DataFrame(filas)
df["valor"] = df["valor"].astype("Int64"); df["costo"] = df["costo"].astype("Int64")
df["tiempo_s"] = df["tiempo_s"].map(lambda x: "—" if pd.isna(x) else f"{x:.3f}")
df.loc[df["valor"].isna(), "metodo"] = "EXACTO (B&B) — no reportado"
display(df.pivot_table(index="metodo", columns="instancia", values=["valor", "tiempo_s"], aggfunc="first")
          [[("valor", "pequena"), ("tiempo_s", "pequena"), ("valor", "comuna"), ("tiempo_s", "comuna"),
            ("valor", "ciudad"), ("tiempo_s", "ciudad")]])

,valor,tiempo_s,valor,tiempo_s,valor,tiempo_s
instancia,pequena,pequena,comuna,comuna,ciudad,ciudad
metodo,,,,,,
EXACTO (B&B),20,0.001,1574,1.150,<NA>,NaN
EXACTO (B&B) — no reportado,<NA>,NaN,<NA>,NaN,<NA>,—
base,15,0.000,1529,0.000,7591,0.004
base + swap,15,0.000,1529,0.001,7630,0.010
eficiencia_marginal,19,0.000,1492,0.000,7706,0.005
eficiencia_marginal + swap,19,0.000,1512,0.001,7784,0.011
hibrida_adaptativa,19,0.000,1529,0.000,7684,0.005
hibrida_adaptativa + swap,19,0.000,1529,0.001,7738,0.016


#### ¿Por qué no se reporta el óptimo en `ciudad`?

In [15]:
print(demostracion_ciudad())

Cada uno de los 80 lotes admite 4 decisiones (no construir, CAI, tránsito o bomberos): 4^80 = 1.462e+48 configuraciones. Aun descartando las que exceden el presupuesto y evaluando 10^9 configuraciones por segundo, enumerarlas tomaría ≈ 1.46e+39 s ≈ 4.63e+31 años (la edad del universo es ≈ 1,4×10^10 años). El Branch & Bound garantiza el óptimo pero su peor caso sigue siendo exponencial, por lo que en `ciudad` solo reportamos heurísticas y búsqueda local, sin garantía de optimalidad.


Con 80 lotes y 4 decisiones por lote hay 4⁸⁰ ≈ 1,46 × 10⁴⁸ configuraciones. Aunque se evaluaran 10⁹ por segundo tardaría ≈ 4,6 × 10³¹ años. Las podas del Branch & Bound ayudan mucho en `comuna` (15 lotes, ~1 s), pero no cambian el carácter exponencial del peor caso.

## Parte 4. Propuesta para el concejo

Elige la propuesta que recomendarías para la instancia `ciudad`, obtenida con cualquiera de tus heurísticas, y preséntala con:

1. **Un mapa** de la ciudad, hecho con `folium` o con cualquier otra librería, donde se distingan los lotes no usados, las estaciones construidas según su tipo, y los incidentes cubiertos y no cubiertos. Mostrar el resultado de forma gráfica sobre un mapa es obligatorio para todos los equipos, participen o no en la feria.
2. **Una tabla** con cada estación construida: lote, tipo, costo y casos que cubre. Al final, el costo total frente al presupuesto, los casos cubiertos de cada tipo y el valor de la propuesta.
3. **Un párrafo** que justifique por qué recomiendas esa propuesta y no otra de las que obtuviste.

Repite el mapa y la tabla para la mejor propuesta posible de la instancia `comuna`.

### Solución Parte 4

**Recomendación para `ciudad`:** se toma la mejor propuesta entre las cuatro heurísticas, refinada con búsqueda local 1-swap. Se genera un mapa `folium` con `LayerControl` (estaciones por tipo, lotes no usados, incidentes cubiertos y no cubiertos y la **subred vial de cobertura**: las calles realmente alcanzadas por el Dijkstra dentro del radio de cada estación) y un panel flotante con KPIs y leyenda. Los mapas se guardan en `salidas/` y se incrustan aquí.

In [16]:
from visual.graficas import mejor_heuristica, curva_pareto, graficar_pareto, analitica_temporal, carga_por_estacion
from visual.mapas import crear_mapa, mapa_calor_no_cubiertos
from visual.informe import parrafo_justificacion

# --- ciudad: mejor heurística (+ swap) ---
al_c = ALC["ciudad"]
nom_c, p_c = mejor_heuristica(al_c)
assert p_c.costo_total <= INST["ciudad"].presupuesto
mapa_c = crear_mapa(al_c, p_c, f"Ciudad – {nom_c} + swap")
mapa_c.save("salidas/mapa_ciudad.html")
print("Propuesta recomendada para ciudad:", nom_c, "+ búsqueda local 1-swap")
print(texto_tabla(al_c, p_c))
IFrame("salidas/mapa_ciudad.html", width="100%", height=650)

Propuesta recomendada para ciudad: eficiencia_marginal + búsqueda local 1-swap
Lote  Tipo        Costo (M)  Casos que cubre  Casos nuevos  Valor
L01   accidente       2.600              234           234    702
L04   hurto             850              355           355    355
L08   hurto           1.000              240           240    240
L11   hurto           1.000              261           261    261
L14   hurto           1.100               84            84     84
L16   accidente       2.100              263           246    738
L19   accidente       1.600              233           203    609
L25   accidente       1.700               89            68    204
L26   accidente       2.800              153            84    252
L28   incendio        4.000               74            74    370
L29   incendio        4.250               64            59    295
L34   hurto           1.400              202           202    202
L35   hurto             800              160           160    1

In [17]:
# --- comuna: óptimo ---
al_m = ALC["comuna"]; p_m = EXACTO["comuna"][0]
crear_mapa(al_m, p_m, "Comuna – propuesta óptima").save("salidas/mapa_comuna_optimo.html")
print(texto_tabla(al_m, p_m))
IFrame("salidas/mapa_comuna_optimo.html", width="100%", height=650)

Lote  Tipo        Costo (M)  Casos que cubre  Casos nuevos  Valor
L03   accidente       1.800              222           222    666
L06   hurto             850              180           180    180
L09   hurto             850              166            95     95
L11   hurto           1.050              200           172    172
L13   accidente       2.100              213            78    234
L14   hurto             800              159            72     72
L15   incendio        4.250               31            31    155
-----------------------------------------------------------------
Costo total: 11.700 M de 12.000 M (sobran 300 M)
Casos cubiertos: hurto 519/562, accidente 300/320, incendio 31/31
Valor de la propuesta: 1574


In [18]:
# Párrafo de justificación (cifras calculadas)
comp_c = {**{k: RES["ciudad"][k].valor for k in HEURISTICAS}}
comp_c[nom_c + " + swap"] = p_c.valor
display(Markdown("**Justificación para el Concejo (ciudad).** " + parrafo_justificacion(al_c, p_c, nom_c + " + swap", comp_c)))
display(Markdown(f"**Comuna (óptimo garantizado).** La propuesta óptima de `comuna` vale {p_m.valor} frente a {RES['comuna']['base'].valor} "
                 f"de la heurística base (+{p_m.valor - RES['comuna']['base'].valor}) con un costo de {p_m.costo_total:,} M de {INST['comuna'].presupuesto:,} M; "
                 f"como el óptimo está certificado por Branch & Bound, ninguna otra propuesta factible la supera.".replace(",", ".")))

**Justificación para el Concejo (ciudad).** Se recomienda la propuesta obtenida con «eficiencia_marginal + swap» para ciudad: 23 estaciones con un costo de 39.950 M de 40.000 M disponibles (99.9 % del presupuesto), que atienden con oportunidad 3764 de 5300 incidentes históricos (71.0 %) y suman un valor ponderado de 7784. Frente a las demás alternativas evaluadas (base: 7591, eficiencia_marginal: 7706, lookahead_1_paso: 7566, hibrida_adaptativa: 7684) es la de mayor valor, y por lo tanto la que más víctimas potenciales protege por peso ponderado (bomberos 5, tránsito 3, CAI 1) sin exceder el presupuesto. Las cifras provienen de un histórico simulado y de una red vial aproximada; en un despliegue real convendría validar los lotes y recalcular con datos actualizados.

**Comuna (óptimo garantizado).** La propuesta óptima de `comuna` vale 1574 frente a 1529 de la heurística base (+45) con un costo de 11.700 M de 12.000 M; como el óptimo está certificado por Branch & Bound. ninguna otra propuesta factible la supera.

## Extras: Pareto, analítica y aplicación interactiva

### Curva de Pareto presupuestal
Valor alcanzable según el presupuesto (de 2.000 a 80.000 millones) con la mejor heurística + swap, y con el óptimo en `comuna`.

In [19]:
pres = list(range(2000, 80001, 2000))
par_m = curva_pareto(ALC["comuna"], pres, con_exacto=True)
par_c = curva_pareto(ALC["ciudad"], pres, con_exacto=False)
fig = graficar_pareto({"comuna": par_m, "ciudad": par_c}); fig.savefig("salidas/pareto.png", dpi=110)
display(fig)
import matplotlib.pyplot as plt; plt.close(fig)

<Figure size 1300x420 with 2 Axes>

### Analítica temporal y auditoría de carga

In [20]:
for n in ("comuna", "ciudad"):
    f = analitica_temporal(INST[n]); display(f); plt.close(f)
f, carga = carga_por_estacion(al_c, p_c); display(f); plt.close(f)
print("Carga (incidentes atendidos por la estación más cercana):", dict(sorted(carga.items(), key=lambda x: -x[1])[:8]), "…")

<Figure size 1600x380 with 3 Axes>

<Figure size 1600x380 with 3 Axes>

<Figure size 1235x400 with 1 Axes>

Carga (incidentes atendidos por la estación más cercana): {'L04': 308, 'L70': 258, 'L39': 255, 'L46': 254, 'L11': 248, 'L08': 228, 'L16': 217, 'L69': 217} …


### Incidentes no cubiertos y 3 nodos candidatos para futuros lotes
Los candidatos se obtienen por cobertura voraz sobre el grafo **transpuesto** (Dijkstra acotado desde cada incidente no cubierto).

In [21]:
m_calor, cands = mapa_calor_no_cubiertos(al_c, p_c, k=3)
m_calor.save("salidas/calor_ciudad.html")
display(pd.DataFrame(cands)[["nodo", "tipo", "valor_adicional", "incidentes", "lat", "lon"]])
IFrame("salidas/calor_ciudad.html", width="100%", height=550)

,nodo,tipo,valor_adicional,incidentes,lat,lon
0,11822,incendio,330,66,6.246387,-75.582687
1,12039,hurto,230,230,6.205039,-75.582099
2,15075,incendio,230,46,6.197402,-75.575427


### Aplicación interactiva para la feria

```bash
pip install -r requirements.txt
streamlit run app_feria_streamlit.py
```

Incluye: selector de instancia **o carga de los 5 archivos para la sustentación en vivo**, controles de presupuesto, **pesos** y **radios**, selección de método (heurísticas, +swap, exacto), mapa interactivo, tabla de estaciones, comparativa, curva de Pareto, analítica, **ruta de despacho calle por calle** desde la estación hasta el incidente y exportación de un **informe HTML**.

In [22]:
print("✔ Todos los casos oficiales (partes 1, 2 y 3) pasaron los assert. Salidas en ./salidas/")

✔ Todos los casos oficiales (partes 1, 2 y 3) pasaron los assert. Salidas en ./salidas/
